> **Work in progress.** This notebook prototypes an analytic, log-space variant of
> Expected Hypervolume Improvement. It runs end to end against the commit above, but
> the implementation is incomplete: it is correct and accurate for cells that lie
> *below* the posterior mean, and it fails in the complementary regime, for the reason
> explained in the last section. It is shared so that others can build on it. Outputs
> are stripped from the committed file.

## Motivation

The LogEI family of acquisition functions addresses a numerical problem rather than a
modeling one. Expected Improvement underflows to exactly zero over most of the search
space in floating point, so its gradient vanishes and the optimizer has nothing to
follow; reformulating the same quantity in log space recovers a usable gradient
essentially everywhere. See
[Ament et al., *Unexpected Improvements to Expected Improvement for Bayesian
Optimization*, NeurIPS 2023](https://arxiv.org/abs/2310.20708).

The multi-objective analytic case has the same problem, and arguably a worse version
of it. Analytic EHVI decomposes the expected hypervolume improvement over the cells of
a box decomposition of the non-dominated region, and within each cell it forms a
*product* of per-outcome factors $\Psi$ and $\nu$ (equations 19 and 25 of
[Yang et al. 2019]). Products of small numbers underflow much faster than individual
small numbers, so the regime where the acquisition value is exactly zero is
correspondingly larger.

BoTorch open-sources the analytic `ExpectedHypervolumeImprovement`, and the Monte
Carlo based `qLogExpectedHypervolumeImprovement` and
`qLogNoisyExpectedHypervolumeImprovement`. What is missing is the analytic function in
log space, which is what this notebook works out.

The bulk of the work is one function: a numerically stable evaluation of
$\log \Psi_{\text{diff}}$, the log of the difference of two $\Psi$ terms. Everything
else follows from it.

In [ ]:
import matplotlib.pyplot as plt
import torch

from botorch.acquisition.analytic import _log_ei_helper
from botorch.acquisition.multi_objective.analytic import (
    ExpectedHypervolumeImprovement,
)
from botorch.utils.probability.utils import log_ndtr, ndtr, phi
from botorch.utils.safe_math import logdiffexp
from botorch.utils.transforms import t_batch_mode_transform
from torch import Tensor

torch.set_default_dtype(torch.double)

## The naive baseline and where it breaks

The quantity of interest is the difference

$$\Psi_{\text{diff}} = \Psi(l, l, \mu, \sigma) - \Psi(l, u, \mu, \sigma),$$

where $l$ and $u$ are the lower and upper bounds of a cell along one outcome. The
definition of $\Psi$ is reproduced below; it matches the `psi` method on the
open-source `ExpectedHypervolumeImprovement`.

In [ ]:
def psi(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    """The Psi function of Yang et al. (2019), eq. 19.

    Psi(l, u, mu, sigma) = sigma * phi((u - mu) / sigma)
                           + (mu - l) * (1 - Phi((u - mu) / sigma))
    """
    u_upper = (upper - mu) / sigma
    return sigma * phi(u_upper) + (mu - lower) * (1 - ndtr(u_upper))


def psi_diff_naive(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    return psi(lower, lower, mu, sigma) - psi(lower, upper, mu, sigma)

In [ ]:
lower = torch.tensor(-1.2)
upper = torch.tensor(0.8)
sigma = torch.tensor(1.0)
mu = torch.linspace(-30.0, 5.0, 351)

naive = psi_diff_naive(lower, upper, mu, sigma)
log_naive = naive.log()

underflowed = ~log_naive.isfinite()
print(f"naive psi_diff underflows to zero at {int(underflowed.sum())} of {len(mu)} points")
if underflowed.any():
    print(f"first underflow at mu = {mu[underflowed][-1].item():.2f}")

## Deriving a stable expression

Write the standardized bounds

$$u_l = \frac{l - \mu}{\sigma}, \qquad u_u = \frac{u - \mu}{\sigma},$$

and let $h(t) = \varphi(t) + t\,\Phi(t)$ be the EI helper, whose logarithm BoTorch
already computes stably as `_log_ei_helper`.

Starting from the definition of $\Psi$ and substituting,

$$\frac{\Psi(l, u, \mu, \sigma)}{\sigma}
   = \varphi(-u_u) - u_l\,\Phi(-u_u).$$

Adding and subtracting $u_u \Phi(-u_u)$ regroups this into the EI helper plus a
remainder:

$$\frac{\Psi(l, u, \mu, \sigma)}{\sigma}
   = \underbrace{\bigl[\varphi(-u_u) - u_u\,\Phi(-u_u)\bigr]}_{h(-u_u)}
     + (u_u - u_l)\,\Phi(-u_u)
   = h(-u_u) + \frac{u - l}{\sigma}\,\Phi(-u_u).$$

The other term is simpler, because $u = l$ collapses the two arguments:

$$\frac{\Psi(l, l, \mu, \sigma)}{\sigma} = h(-u_l).$$

So the difference is

$$\frac{\Psi_{\text{diff}}}{\sigma}
   = h(-u_l) - \left[ h(-u_u) + \frac{u-l}{\sigma}\,\Phi(-u_u) \right],$$

which is a difference of two positive quantities. Taking logarithms and using
`logsumexp` for the bracketed sum and `logdiffexp` for the outer subtraction:

$$\log \Psi_{\text{diff}} = \log \sigma
  + \operatorname{logdiffexp}\Bigl(
      \operatorname{logsumexp}\bigl[\log h(-u_u),\;
        \log\tfrac{u-l}{\sigma} + \log\Phi(-u_u)\bigr],\;
      \log h(-u_l)
    \Bigr).$$

Every term on the right is computed by an existing stable primitive: `_log_ei_helper`
for $\log h$, `log_ndtr` for $\log \Phi$, and `logsumexp` / `logdiffexp` from
`botorch.utils.safe_math`. Note that $\log\Phi(-u_u)$ is evaluated directly through
`log_ndtr` rather than as $\log(1 - \Phi(u_u))$, which would lose all precision in
exactly the tail regime we care about.

In [ ]:
def log_psi_diff(
    lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor
) -> Tensor:
    """Numerically stable evaluation of log(psi(l, l, mu, s) - psi(l, u, mu, s)).

    Args:
        lower: `num_cells x m`-dim tensor of lower cell bounds.
        upper: `num_cells x m`-dim tensor of upper cell bounds.
        mu: `batch_shape x 1 x m`-dim tensor of means.
        sigma: `batch_shape x 1 x m`-dim tensor of standard deviations.

    Returns:
        A `batch_shape x num_cells x m`-dim tensor of log values.
    """
    u_lower = (lower - mu) / sigma
    u_upper = (upper - mu) / sigma
    log_u_diff = ((upper - lower) / sigma).log()

    # log of h(-u_upper) + ((u - l) / sigma) * Phi(-u_upper)
    log_subtrahend = torch.logsumexp(
        torch.stack(
            [
                _log_ei_helper(-u_upper),
                log_u_diff + log_ndtr(-u_upper),
            ],
            dim=0,
        ),
        dim=0,
    )
    # log of h(-u_lower), the larger of the two terms
    log_minuend = _log_ei_helper(-u_lower)

    return sigma.log() + logdiffexp(log_a=log_subtrahend, log_b=log_minuend)

### Accuracy, not just finiteness

The interesting failure is not that the naive form underflows. It is that it becomes
*silently wrong* well before it underflows, and returns a plausible-looking finite
number while doing so.

The cause is the `1 - Phi(u)` in the definition of $\Psi$. Once
$1 - \Phi(u) \lesssim \varepsilon_{\text{machine}}$, the subtraction rounds to exactly
zero, so $\Psi(l, l, \mu, \sigma)$ collapses to its $\sigma\varphi$ term and the
$(\mu - l)(1 - \Phi)$ term that was cancelling most of it disappears. What comes back
is not a slightly perturbed answer; it is a different quantity.

Comparing both against a high-precision reference makes the size of the effect clear.

In [ ]:
# Reference values of log(psi_diff), computed at 200 decimal digits with mpmath.
# The complementary error function is used for 1 - Phi, since computing it as a
# subtraction loses all precision in exactly the regime of interest.
#
#     mp.mp.dps = 200
#     def mp_psi(l, u, mu, s):
#         uu = (u - mu) / s
#         pdf = mp.e ** (-uu**2 / 2) / mp.sqrt(2 * mp.pi)
#         return s * pdf + (mu - l) * (mp.erfc(uu / mp.sqrt(2)) / 2)
#     mp.log(mp_psi(l, l, mu, s) - mp_psi(l, u, mu, s))
#
REFERENCE = {  # mu -> log(psi_diff), for lower = -1.2, upper = 0.8, sigma = 1
    -2: -2.1686147892,
    -4: -7.1830521292,
    -6: -15.6907858156,
    -8: -27.9333400888,
    -10: -44.0255640224,
    -12: -64.0230195528,
    -15: -101.4037487309,
    -20: -183.5150577461,
    -30: -422.3632910630,
    -50: -1199.4156570546,
    -75: -2732.7422064614,
    -100: -4890.8254409643,
}

print(f"{'mu':>6} | {'exact':>16} | {'log(naive)':>16} | {'err':>9} "
      f"| {'log_psi_diff':>16} | {'err':>9}")
print("-" * 82)
for mu_value, exact in REFERENCE.items():
    m = torch.tensor(float(mu_value))
    naive_value = psi_diff_naive(lower, upper, m, sigma).log().item()
    stable_value = log_psi_diff(lower, upper, m, sigma).item()
    naive_err = abs(naive_value - exact)
    stable_err = abs(stable_value - exact)
    print(
        f"{mu_value:>6} | {exact:>16.6f} | {naive_value:>16.6f} | {naive_err:>9.1e} "
        f"| {stable_value:>16.6f} | {stable_err:>9.1e}"
    )

In [ ]:
mu_plot = torch.linspace(-30.0, 12.0, 421)
naive_plot = psi_diff_naive(lower, upper, mu_plot, sigma).log()
stable_plot = log_psi_diff(lower, upper, mu_plot, sigma)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mu_plot, stable_plot, lw=2, label="log_psi_diff")
ax.plot(mu_plot, naive_plot, lw=2, ls=":", label="log(psi_diff), naive")
ax.scatter(
    list(REFERENCE), list(REFERENCE.values()),
    color="black", zorder=5, s=28, label="high-precision reference",
)
ax.axvspan(lower.item(), upper.item(), color="tab:green", alpha=0.12)
ax.text(
    (lower + upper).item() / 2, -60, "cell", ha="center",
    color="tab:green", fontsize=9,
)
ax.set_xlabel("mu")
ax.set_ylabel("log psi_diff")
ax.set_ylim(-450, 20)
ax.set_title(
    "Below the cell the naive form departs from the truth, then underflows.\\n"
    "Above the cell both forms fail."
)
ax.legend(loc="lower right")

### Gradients, and the limits of the single-branch implementation

Finite values are not enough: an expression can return a finite result and still emit
`NaN` gradients, usually because a masked-out branch of a `torch.where` is evaluated
anyway and its derivative is undefined.

Checking across regimes also shows where this implementation stops working, which is
more useful than a single pass/fail. There are three regimes to distinguish: below the
cell (the case the log form is designed for), above the cell (where the two terms of
the difference converge and cancel), and cells that are narrow relative to `sigma`.

In [ ]:
def check_gradients(dtype: torch.dtype, sigma_value: float, lo: float, hi: float):
    n = 200
    mu_g = torch.linspace(lo, hi, n, dtype=dtype, requires_grad=True)
    sigma_g = torch.full((n,), sigma_value, dtype=dtype, requires_grad=True)
    lower_g = torch.tensor(-1.2, dtype=dtype)
    upper_g = torch.tensor(0.8, dtype=dtype)

    value = log_psi_diff(lower_g, upper_g, mu_g, sigma_g)
    value.sum().backward()

    def n_bad(t: Tensor) -> int:
        return int((t.isnan() | t.isinf()).sum())

    return n_bad(value), n_bad(mu_g.grad), n_bad(sigma_g.grad)


regimes = [
    ("below the cell, moderate", 1.0, -8.0, -2.0),
    ("below the cell, deep", 1.0, -30.0, -10.0),
    ("spans the cell", 1.0, -6.0, 6.0),
    ("narrow cell, (u - l) / sigma = 2e-4", 1e4, 0.0, 100.0),
]

print(f"{'regime':>38} | {'dtype':>7} | {'value':>6} | {'d/dmu':>6} | {'d/dsigma':>9}")
print("-" * 82)
for label, sigma_value, lo, hi in regimes:
    for dtype in (torch.float32, torch.float64):
        bad = check_gradients(dtype, sigma_value, lo, hi)
        name = "float32" if dtype is torch.float32 else "float64"
        print(
            f"{label:>38} | {name:>7} | " + " | ".join(f"{b:>6}" for b in bad[:2])
            + f" | {bad[2]:>9}"
        )

Reading the table: in double precision the implementation is clean below and across the
cell, and the only failures come from cells that are narrow relative to `sigma`, which
is the missing Taylor branch described at the end. In single precision that branch
matters much sooner. The `spans the cell` row picks up the above-cell failure, which is
a genuine gap in the derivation rather than a precision issue, and is also discussed at
the end.

## The $\nu$ term

The second factor is straightforward, since it is already a product rather than a
difference — taking logs turns it into a sum with no cancellation.

$$\log \nu(l, u, \mu, \sigma) = \log(u - l) + \log \Phi\!\left(\frac{\mu - u}{\sigma}\right).$$

In [ ]:
def log_nu(lower: Tensor, upper: Tensor, mu: Tensor, sigma: Tensor) -> Tensor:
    """Logarithm of the Nu function of Yang et al. (2019), eq. 25."""
    return log_ndtr((mu - upper) / sigma) + (upper - lower).log()


def check_log_nu_gradients(dtype: torch.dtype, sigma_value: float) -> None:
    n = 200
    mu_g = torch.linspace(-30.0, 30.0, n, dtype=dtype, requires_grad=True)
    sigma_g = torch.full((n,), sigma_value, dtype=dtype, requires_grad=True)
    value = log_nu(
        torch.tensor(-1.2, dtype=dtype), torch.tensor(0.8, dtype=dtype), mu_g, sigma_g
    )
    value.sum().backward()
    bad = [
        int((t.isnan() | t.isinf()).sum()) for t in (value, mu_g.grad, sigma_g.grad)
    ]
    name = "float32" if dtype is torch.float32 else "float64"
    print(f"log_nu, {name}, sigma={sigma_value:>6g}: value/dmu/dsigma bad = {bad}")


for dtype in (torch.float32, torch.float64):
    for sigma_value in (1.0, 1e4):
        check_log_nu_gradients(dtype, sigma_value)

## Assembling the acquisition function

With both factors available in log space, the rest of the analytic EHVI computation
carries over. For each cell the contribution is a product over outcomes of either
$\Psi_{\text{diff}}$ or $\nu$, taken over all $2^m$ combinations; in log space the
product becomes a sum, and the outer sums over cells and combinations become
`logsumexp`.

Subclassing the open-source `ExpectedHypervolumeImprovement` means the partitioning and
cell-bound bookkeeping is inherited, and only `forward` needs to change.

In [ ]:
class LogExpectedHypervolumeImprovement(ExpectedHypervolumeImprovement):
    r"""Analytic log Expected Hypervolume Improvement.

    Computes the same quantity as `ExpectedHypervolumeImprovement`, but in log space,
    so that the value and its gradient remain informative in regimes where the
    non-log version underflows to exactly zero.

    Supports `q = 1` and no pending points, matching the analytic parent class.
    """

    @t_batch_mode_transform()
    def forward(self, X: Tensor) -> Tensor:
        posterior = self.model.posterior(
            X, posterior_transform=self.posterior_transform
        )
        mu = posterior.mean
        sigma = posterior.variance.clamp_min(1e-9).sqrt()

        # The upper bounds contain infs, which are not differentiable.
        cell_upper_bounds = self.cell_upper_bounds.clamp_max(
            1e10 if X.dtype == torch.double else 1e8
        )

        log_psi_diff_values = log_psi_diff(
            lower=self.cell_lower_bounds,
            upper=cell_upper_bounds,
            mu=mu,
            sigma=sigma,
        )
        log_nu_values = log_nu(
            lower=self.cell_lower_bounds,
            upper=cell_upper_bounds,
            mu=mu,
            sigma=sigma,
        )

        # batch_shape x num_cells x 2 x m
        stacked_factors = torch.stack([log_psi_diff_values, log_nu_values], dim=-2)

        # Take the cross product across outcomes: batch_shape x num_cells x 2^m x m
        all_log_factors = stacked_factors.gather(
            dim=-2,
            index=self._cross_product_indices.expand(
                stacked_factors.shape[:-2] + self._cross_product_indices.shape
            ),
        )

        # Product over outcomes becomes a sum in log space; the sums over cells and
        # over the cross product become logsumexp.
        return torch.logsumexp(all_log_factors.sum(dim=-1), dim=(-2, -1))

### Comparison against the non-log version

Two things to separate here. Where EHVI is comfortably positive the two should agree
after exponentiation. Where EHVI has gone to exactly zero, LogEHVI should still return
a finite value with a usable gradient.

There is also a middle band, and it is the interesting one: the cell-level table above
showed that the naive `psi_diff` is *wrong*, not merely small, before it underflows.
That error propagates into EHVI, so a disagreement between the two in that band is not
evidence that LogEHVI is wrong. The comparison below therefore reports agreement as a
function of the magnitude of EHVI, rather than as a single worst-case number.

In [ ]:
from botorch.models import SingleTaskGP
from botorch.models.model_list_gp_regression import ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.test_functions.multi_objective import BraninCurrin
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from botorch.utils.sampling import draw_sobol_samples
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood

from botorch.fit import fit_gpytorch_mll

torch.manual_seed(0)

problem = BraninCurrin(negate=True)
d, m = problem.dim, problem.num_objectives
noise_se = torch.full((m,), 1e-2)

train_X = draw_sobol_samples(bounds=problem.bounds, n=16, q=1).squeeze(1)
train_Y = problem(train_X) + torch.randn(16, m) * noise_se

models = [
    SingleTaskGP(
        train_X=train_X,
        train_Y=train_Y[..., i : i + 1],
        train_Yvar=torch.full_like(train_Y[..., i : i + 1], noise_se[i] ** 2),
        input_transform=Normalize(d=d, bounds=problem.bounds),
        outcome_transform=Standardize(m=1),
    )
    for i in range(m)
]
model = ModelListGP(*models)
fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))

with torch.no_grad():
    pred = model.posterior(train_X).mean
partitioning = FastNondominatedPartitioning(ref_point=problem.ref_point, Y=pred)

ehvi = ExpectedHypervolumeImprovement(
    model=model, ref_point=problem.ref_point, partitioning=partitioning
)
log_ehvi = LogExpectedHypervolumeImprovement(
    model=model, ref_point=problem.ref_point, partitioning=partitioning
)

In [ ]:
test_X = draw_sobol_samples(bounds=problem.bounds, n=512, q=1).squeeze(1)

with torch.no_grad():
    ehvi_values = ehvi(test_X.unsqueeze(-2))
    log_ehvi_values = log_ehvi(test_X.unsqueeze(-2))

positive = ehvi_values > 0
print(f"EHVI is exactly zero at {int((~positive).sum())} of {len(test_X)} test points")

log_from_ehvi = ehvi_values[positive].log()
log_direct = log_ehvi_values[positive]
discrepancy = (log_direct - log_from_ehvi).abs()

print(f"\n{'log(EHVI) range':>22} | {'n':>5} | {'max |difference| in log space':>30}")
print("-" * 64)
bands = [(0.0, float("inf")), (-5.0, 0.0), (-15.0, -5.0),
         (-40.0, -15.0), (float("-inf"), -40.0)]
for lo, hi in bands:
    in_band = (log_from_ehvi > lo) & (log_from_ehvi <= hi)
    if not in_band.any():
        continue
    label = f"({lo:g}, {hi:g}]"
    print(f"{label:>22} | {int(in_band.sum()):>5} | {discrepancy[in_band].max():>30.2e}")

print(
    "\nAgreement is tight while EHVI is large and degrades as it shrinks, which is the"
    "\nsignature of the naive cancellation rather than of an error in the log form."
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
order = ehvi_values.argsort()
ax.plot(
    log_ehvi_values[order], lw=1.5, label="LogEHVI",
)
with torch.no_grad():
    naive_log = ehvi_values[order].log()
ax.plot(naive_log, lw=1.5, ls=":", label="log(EHVI)")
ax.set_xlabel("test point (sorted by EHVI)")
ax.set_ylabel("log acquisition value")
ax.set_title("log(EHVI) truncates at -inf; LogEHVI keeps going")
ax.legend()

In [ ]:
# Gradients where EHVI has underflowed: LogEHVI should still provide a direction.
if (~positive).any():
    X_zero = test_X[~positive][:8].clone().requires_grad_(True)

    ehvi_grad_value = ehvi(X_zero.unsqueeze(-2)).sum()
    ehvi_grad_value.backward()
    ehvi_grad_norm = X_zero.grad.norm(dim=-1)

    X_zero_log = test_X[~positive][:8].clone().requires_grad_(True)
    log_ehvi(X_zero_log.unsqueeze(-2)).sum().backward()
    log_grad_norm = X_zero_log.grad.norm(dim=-1)

    print(f"{'|grad EHVI|':>14} | {'|grad LogEHVI|':>16}")
    for a, b in zip(ehvi_grad_norm, log_grad_norm):
        print(f"{a.item():>14.3e} | {b.item():>16.3e}")

### The regime this implementation does not handle

For completeness, the failure mode. When the posterior mean sits *above* a cell, the
two terms of $\Psi_{\text{diff}}$ converge towards one another: as $\mu$ grows,
$h(-u_l) \to \mu - l$ and the bracketed subtrahend $\to \mu - l$ as well. Their
difference is then a subtraction of two nearly equal logarithms, and `logdiffexp`
returns $-\infty$ once they agree to machine precision.

This is not a precision problem that more digits would fix — it is the wrong algebraic
arrangement for that regime, and it needs a separate branch that groups the terms so
the cancellation is explicit. The naive form happens to fail at roughly the same place
here, so nothing is lost relative to the status quo, but nothing is gained either.

In [ ]:
mu_above = torch.tensor([2.0, 5.0, 8.0, 10.0, 15.0, 30.0])
print(f"{'mu':>6} | {'log(naive)':>13} | {'log_psi_diff':>14}")
print("-" * 40)
for m in mu_above:
    naive_v = psi_diff_naive(lower, upper, m, sigma).log().item()
    stable_v = log_psi_diff(lower, upper, m, sigma).item()
    print(f"{m.item():>6.0f} | {naive_v:>13.6f} | {stable_v:>14.6f}")
print("\nBoth forms lose the value above mu ~ 10 for this cell; see the notes below.")

## Status & open problems

What works: below and across the cell, in double precision, `log_psi_diff` matches a
200-digit reference to roughly `1e-11` in log space over the whole range tested, out to
`log psi_diff` around `-4900`, where the naive form has been returning zero for a long
time and returning wrong non-zero values for a while before that. Assembled into
`LogExpectedHypervolumeImprovement`, that carries through to finite acquisition values
and non-zero gradients where analytic EHVI provides neither.

What does not:

- **The above-cell branch is missing.** Demonstrated above. The fix is to regroup the
  difference so that the cancelling terms are combined analytically rather than
  numerically, using `log_prob_normal_in` for the $\Phi$ difference; a prototype
  existed but was never masked correctly, and an unmasked branch will reintroduce
  `NaN` gradients on the points it does not apply to. The masking pattern that works is
  to `masked_fill` the inactive branch *before* the `torch.where`, not after.
- **The narrow-cell branch is missing.** When $(u-l)/\sigma$ is small the two $\Psi$
  terms nearly cancel and precision is lost — visible in the single-precision rows of
  the gradient table. A Taylor expansion in $(u-l)/\sigma$ is the right tool, with
  leading behaviour
  $\log \Psi_{\text{diff}} \approx \log h(-u_l) + \log(1 - e^{\kappa})$,
  $\kappa = 2\log\frac{u-l}{\sigma} + \log\varphi(-u_l) - \log h(-u_l) - \log 2$.
  Its accuracy was never characterised and the crossover point never chosen. The likely
  shape of the fix is a two-branch structure of the kind inside `_log_ei_helper`.
- **$\Psi_{\text{diff}}$ is not symmetric about the cell centre**, which is what makes a
  single unified branch awkward and is worth bearing in mind when designing the
  crossover logic.
- **No `q`-batch support.** This inherits the `q = 1` restriction of the analytic parent
  class. Whether the analytic factors compose with the machinery in `qLogEHVI`, or
  whether the analytic route is simply the wrong tool for batches, is open.
- **No closed-loop benchmark** against `qLogExpectedHypervolumeImprovement` or
  `qLogNParEGO`. Given the `q = 1` restriction, the honest question is whether an
  analytic version earns its place at all once the Monte Carlo ones exist; the case
  would be lower variance and lower cost in the sequential setting, and that has not
  been demonstrated.

A separate observation that came out of this and is worth stating on its own: the
`1 - Phi` cancellation is in the **released** analytic `ExpectedHypervolumeImprovement`,
not only in the naive helper written here. In the band before it underflows, it returns
values that are too large by orders of magnitude. That matters little for optimization,
since it happens where the acquisition value is negligible, but it is worth knowing when
EHVI values are used for anything other than argmax.

---

## References

- S. Ament, S. Daulton, D. Eriksson, M. Balandat, E. Bakshy.
  [Unexpected Improvements to Expected Improvement for Bayesian Optimization](https://arxiv.org/abs/2310.20708).
  Advances in Neural Information Processing Systems 36, 2023.
- K. Yang, M. Emmerich, A. Deutz, T. Bäck. Efficient Computation of Expected
  Hypervolume Improvement Using Box Decomposition Algorithms. Journal of Global
  Optimization, 2019.
- S. Daulton, M. Balandat, E. Bakshy. Differentiable Expected Hypervolume Improvement
  for Parallel Multi-Objective Bayesian Optimization. Advances in Neural Information
  Processing Systems 33, 2020.